# 13 - Correspondencia entre operación y perfiles de propiedades

- Pregunta: ¿las etiquetas existentes de **venta**, **alquiler permanente** y **alquiler temporal** se asocian con grupos que emergen al agrupar publicaciones por atributos de la propiedad y su entorno?
- Unidad: publicación inmobiliaria; el clustering no recibe el tipo de operación ni la fuente.
- Se fija `k=3` por las tres etiquetas preexistentes. No se pretende descubrir ni definir nuevas categorías oficiales.
- PCA representa variables numéricas comparables; K-means genera una partición no supervisada que luego se contrasta con las etiquetas.
- La asociación principal se contrasta con permutaciones de etiquetas dentro de estratos fuente × barrio que contienen más de una operación, para reducir el efecto de la composición de portales.
- Precio no entra en el espacio principal: venta (stock/precio total), alquiler (flujo mensual) y temporario tienen unidades o coberturas distintas. Se resume después, por operación y por separado.
- Insumo estable: `data/processed/df_publicaciones_enriquecidas_v1.csv`; contexto barrial: `kpis_barrio_v1.csv`.

- En cada bloque se declaran los supuestos y límites que afectan su lectura; las conclusiones quedan condicionadas a la cobertura y al corte de publicaciones disponibles.


## 1. Alcance, hipótesis y controles

- **H0:** dentro de combinaciones fuente × barrio que contienen más de una operación, la etiqueta de operación no muestra asociación con el cluster, más allá de lo esperable al permutar etiquetas conservando la composición de cada fuente y barrio.
- **H1:** la asociación observada supera ese nulo condicionado. Se reportan magnitud, tabla de composición, ARI/AMI, V de Cramér e incertidumbre descriptiva; no se interpreta como causalidad.
- Variables candidatas: ambientes, superficie y atributos físicos con cobertura suficiente; distancia a transporte/espacio verde; densidades barriales de servicios y porcentaje verde.
- Se excluyen identificadores, precio, operación, fuente y flags de calidad del espacio de clustering. La fuente se usa solo para el contraste condicionado.
- Se excluyen registros marcados como presentes en varias operaciones para evitar contar el mismo identificador con etiquetas diferentes.
- Los resultados dependen de publicaciones observadas y geocodificadas: no representan todas las viviendas ni transacciones.

- **Supuestos y salvedades:** la unidad es una publicación capturada, no un inmueble único ni una transacción. Excluir IDs marcados en más de una operación reduce duplicación etiquetada, pero no resuelve avisos distintos de una misma propiedad ni diferencias en la recolección de cada portal.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, adjusted_mutual_info_score, normalized_mutual_info_score
from src import config

REPORTS = config.REPORTS_DIR
base = pd.read_csv(config.PROCESSED_DIR / "df_publicaciones_enriquecidas_v1.csv", low_memory=False, dtype={"id_publicacion": str})
kb = pd.read_csv(config.PROCESSED_DIR / "kpis_barrio_v1.csv")

# Normalizar alias sin reclasificar operaciones cuya naturaleza no sea explícita.
op = base["tipo_operacion_norm"].astype(str).str.lower().str.strip()
alias = {
    "alquiler_temporario": "alquiler_temporal",
    "alquiler temporal": "alquiler_temporal", "temporario": "alquiler_temporal",
    "alquiler_temp": "alquiler_temporal", "temporary_rent": "alquiler_temporal",
    "alquiler_permanente": "alquiler", "alquiler permanente": "alquiler",
}
base["segmento_operacion"] = op.replace(alias)
base["segmento_operacion"] = base["segmento_operacion"].replace({"alquiler": "alquiler_permanente"})
segmentos = ["venta", "alquiler_permanente", "alquiler_temporal"]
muestra = base[base["segmento_operacion"].isin(segmentos)].copy()

# Una observación repetida entre operaciones puede hacer que el algoritmo aprenda la etiqueta duplicada.
if "id_en_multiples_operaciones_flag" in muestra:
    multi = muestra["id_en_multiples_operaciones_flag"].astype(str).str.lower().isin(["true", "1", "si", "sí"])
    muestra = muestra.loc[~multi].copy()

# Incorporar contexto por barrio si no está ya materializado en la base enriquecida.
ctx_candidatas = [c for c in kb.columns if c.startswith("densidad_") or c == "pct_superficie_verde"]
ctx_disponibles = [c for c in ctx_candidatas if c not in muestra.columns]
if ctx_disponibles:
    muestra = muestra.merge(kb[["barrio_oficial"] + ctx_disponibles], on="barrio_oficial", how="left", validate="many_to_one")

print("Publicaciones por operación antes del filtro de cobertura:")
display(muestra["segmento_operacion"].value_counts().reindex(segmentos, fill_value=0).rename("n").to_frame())
print("Fuentes:", muestra["fuente_norm"].nunique(), "| Barrios:", muestra["barrio_oficial"].nunique())


Publicaciones por operación antes del filtro de cobertura:


,n
segmento_operacion,
venta,43232
alquiler_permanente,9333
alquiler_temporal,12863


Fuentes: 4 | Barrios: 48


## 2. Selección transparente de variables y cobertura

- La cobertura se calcula por segmento antes de fijar la muestra analítica.
- Se conservan variables numéricas observadas/derivadas con al menos 60% de datos en **cada** segmento; la regla evita que la PCA represente solo un portal o una modalidad.
- Después se usa el conjunto completo de casos en esas variables. No se imputa con medias de otros segmentos.
- Los conteos, densidades, distancias y superficies se transforman con `log1p` para reducir asimetría; ambientes, baños y porcentaje verde quedan en su escala original antes del estandarizado.
- La lista final debe revisarse con la tabla de cobertura: si quedan muy pocas variables o la muestra pierde demasiados casos, el resultado se informa como no evaluable y se ajusta la regla con justificación.

### Supuestos y límites de la selección

- La regla de cobertura es operativa, no demuestra que los faltantes sean aleatorios. La muestra final representa casos completos dentro de las fuentes y segmentos que superan el umbral.
- Evitar imputaciones reduce perfiles artificiales; a cambio puede cambiar la composición de la muestra. Revisar cobertura y n por segmento antes de interpretar.
- Ambientes, baños, dormitorios y superficie pueden estar reconstruidos o medidos de forma distinta por portal; son atributos publicados armonizados, no mediciones homogéneas certificadas.


In [ ]:
candidatas = [
    "ambientes_ref", "dormitorios_ref", "banios_ref", "cocheras_ref", "superficie_ref_m2",
    "dist_transporte_masivo_m", "dist_espacio_verde_m", "dist_hospitales_m",
    "densidad_subte_km2", "densidad_tren_km2", "densidad_metrobus_km2",
    "densidad_educacion_km2", "densidad_hospitales_km2", "densidad_culturales_km2",
    "densidad_gastronomia_km2", "pct_superficie_verde",
]
candidatas = [c for c in candidatas if c in muestra.columns]
cobertura = muestra.groupby("segmento_operacion")[candidatas].apply(lambda x: x.notna().mean()).T
cobertura.columns.name = "segmento_operacion"
display((100 * cobertura).round(1).rename_axis("variable").reset_index())
min_cobertura = 0.60
features = [c for c in candidatas if (cobertura.loc[c] >= min_cobertura).all()]
print("Variables que cumplen cobertura mínima:", features)
assert len(features) >= 3, "No hay 3 variables con cobertura suficiente; revisar cobertura/alcance antes del clustering."

analisis = muestra.dropna(subset=features + ["fuente_norm", "segmento_operacion"]).copy()
print("Casos completos:", len(analisis), "de", len(muestra), "({:.1f}%)".format(100 * len(analisis) / max(1, len(muestra))))
display(analisis.groupby("segmento_operacion").size().reindex(segmentos, fill_value=0).rename("n_completo").to_frame())
assert analisis["segmento_operacion"].nunique() == 3, "La muestra completa no contiene las tres operaciones."

X = analisis[features].apply(pd.to_numeric, errors="coerce")
assert X.notna().all().all(), "La selección de casos completos debe preceder al escalado."
cols_log = [c for c in features if any(t in c for t in ["superficie", "dist_", "densidad_", "cocheras", "dormitorios", "banios", "ambientes"])]
for c in cols_log:
    if X[c].min() >= 0:
        X[c] = np.log1p(X[c])
Xz = StandardScaler().fit_transform(X)
y = analisis["segmento_operacion"].to_numpy()
print("Dimensión de la matriz estandarizada:", Xz.shape)


## 3. PCA y lectura de los ejes

- PCA se ajusta sobre las publicaciones completas usando solo las variables seleccionadas, sin etiquetas ni precio.
- Se conserva el menor número de componentes que alcanza 80% de varianza explicada, con máximo de 8; K-means trabaja sobre esos componentes.
- La varianza y las cargas describen el espacio; no convierten los ejes en dimensiones habitacionales oficiales.

### Supuestos y límites de PCA

- PCA resume asociaciones lineales en variables estandarizadas y transformadas; los componentes dependen de la selección de variables y pueden cambiar ante otra regla de cobertura.
- La varianza explicada no prueba que los ejes sean útiles para clasificar operaciones. Componentes ortogonales son una propiedad matemática, no dimensiones sociales independientes.


In [ ]:
pca = PCA().fit(Xz)
coord = pca.transform(Xz)
varianza = pd.DataFrame({"componente": ["PC{}".format(i + 1) for i in range(len(pca.explained_variance_ratio_))],
                         "varianza_explicada": pca.explained_variance_ratio_,
                         "varianza_acumulada": np.cumsum(pca.explained_variance_ratio_)})
n_pc = min(int(np.searchsorted(varianza["varianza_acumulada"].to_numpy(), .80) + 1), 8, coord.shape[1])
Z = coord[:, :n_pc]
cargas = pd.DataFrame(pca.components_[:n_pc].T, index=features,
                      columns=["PC{}".format(i + 1) for i in range(n_pc)])
display(varianza.round(3))
display(cargas.round(3))
varianza.to_csv(REPORTS / "pe3_pca_operaciones_varianza_v1.csv", index=False)
cargas.to_csv(REPORTS / "pe3_pca_operaciones_cargas_v1.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(varianza["componente"], 100 * varianza["varianza_explicada"], color="#4b78a8")
ax.plot(varianza["componente"], 100 * varianza["varianza_acumulada"], color="#c45a46", marker="o")
ax.axhline(80, color="#777", linestyle="--", linewidth=1)
ax.set_ylabel("Varianza explicada (%)")
ax.set_title("PCA de atributos comparables: varianza explicada y acumulada")
plt.tight_layout(); plt.show()


## 4. Clustering no supervisado con k=3

- `k=3` se define por la cantidad de etiquetas existentes, no por maximizar su coincidencia.
- La silueta se informa para k=2–6 como diagnóstico geométrico, sin seleccionar k por su alineación con operación.
- Se valida estabilidad del K-means de k=3 mediante 30 submuestras del 80% y ARI contra la asignación base en los casos compartidos.
- Una asociación alta no implica que las operaciones sean categorías naturales de vivienda: puede reflejar variables de publicación, cobertura o geografía.

### Supuestos y límites de K-means

- K-means usa distancia euclídea y favorece grupos compactos, comparables en escala y aproximadamente convexos. Puede ser sensible a inicialización, valores extremos y al predominio de atributos repetidos por barrio.
- Fijar k=3 responde a la comparación con tres etiquetas existentes; no demuestra que haya tres grupos naturales. Silueta y ARI de submuestras son diagnósticos complementarios, no validación sustantiva.
- Los registros del mismo barrio comparten indicadores contextuales. La prueba posterior conserva composición por fuente y barrio, pero la partición sigue describiendo esta muestra de avisos.


In [ ]:
rng = np.random.default_rng(2026)
n = len(Z)
sample_sil = min(4000, n)
validacion_interna = []
modelos = {}
for k in range(2, min(6, n - 1) + 1):
    km = KMeans(n_clusters=k, n_init=30, random_state=2026).fit(Z)
    modelos[k] = km
    sil = silhouette_score(Z, km.labels_, sample_size=sample_sil, random_state=2026) if n > sample_sil else silhouette_score(Z, km.labels_)
    validacion_interna.append({"k": k, "silueta_muestra": sil})
validacion_interna = pd.DataFrame(validacion_interna)
display(validacion_interna.round(3))

k = 3
modelo = modelos[k]
cluster = modelo.labels_ + 1
analisis["cluster_propiedad"] = cluster
ari_estabilidad = []
for rep in range(30):
    idx = np.sort(rng.choice(n, size=max(k + 2, int(.8 * n)), replace=False))
    sub = KMeans(n_clusters=k, n_init=20, random_state=rep).fit_predict(Z[idx])
    ari_estabilidad.append(adjusted_rand_score(cluster[idx], sub))
estabilidad = pd.Series(ari_estabilidad)
print("ARI de estabilidad, mediana:", round(estabilidad.median(), 3),
      "| rango intercuartílico:", tuple(np.round(estabilidad.quantile([.25, .75]).to_numpy(), 3)))

var_label = pd.crosstab(analisis["cluster_propiedad"], analisis["segmento_operacion"], normalize="index").reindex(columns=segmentos, fill_value=0) * 100
n_cluster = analisis["cluster_propiedad"].value_counts().sort_index().rename("publicaciones")
composicion = var_label.join(n_cluster)
display(composicion.round(1))
composicion.to_csv(REPORTS / "pe3_clusters_propiedades_composicion_v1.csv")
validacion_interna.to_csv(REPORTS / "pe3_clusters_propiedades_silueta_v1.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4.5))
for op_label, grupo in analisis.groupby("segmento_operacion"):
    ix = grupo.index
    loc = analisis.index.get_indexer(ix)
    ax.scatter(coord[loc, 0], coord[loc, 1], s=9, alpha=.35, label=op_label)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Publicaciones en PC1–PC2: color por etiqueta de operación")
ax.legend(frameon=False, markerscale=2)
plt.tight_layout(); plt.show()


## 5. Contraste de correspondencia con las etiquetas

- ARI y AMI comparan la partición con las tres etiquetas; V de Cramér resume asociación en la tabla cluster × operación.
- La prueba de permutación mantiene fijos los clusters y permuta etiquetas **dentro de cada fuente**, en los estratos que tienen dos o más operaciones observadas. Así se pregunta si la correspondencia se sostiene dentro de portales con variación real de operación.
- Fuentes con una sola operación no identifican asociación condicionada y quedan fuera de este contraste, aunque sí aparecen en los resultados descriptivos generales.
- El p-valor de permutación se acompaña de efecto observado; no se toma significancia como evidencia de importancia práctica.

### Supuestos y límites de la permutación

- El nulo requiere que, dentro de cada estrato fuente × barrio con varias operaciones, las etiquetas sean intercambiables bajo ausencia de asociación residual. No corrige diferencias de fecha, geocodificación, perfil de vendedor, selección de avisos ni medición de atributos.
- La prueba condicionada solo habla de los estratos con solapamiento de modalidades; si quedan pocos barrios o no hay casos temporarios en ellos, no permite generalizar a todas las publicaciones.
- Puede persistir dependencia entre avisos de una inmobiliaria o una propiedad. El p-valor es una referencia de aleatorización condicionada, no un intervalo de confianza poblacional; se priorizan tamaño, composición y estabilidad.


In [ ]:
def cramers_v(tabla):
    obs = tabla.to_numpy(dtype=float)
    total = obs.sum()
    if total == 0 or min(obs.shape) < 2:
        return np.nan
    esperado = np.outer(obs.sum(axis=1), obs.sum(axis=0)) / total
    chi2 = np.nansum((obs - esperado) ** 2 / np.where(esperado == 0, np.nan, esperado))
    phi2 = chi2 / total
    r, c = obs.shape
    phi2_corr = max(0, phi2 - ((c - 1) * (r - 1)) / max(total - 1, 1))
    r_corr = r - ((r - 1) ** 2) / max(total - 1, 1)
    c_corr = c - ((c - 1) ** 2) / max(total - 1, 1)
    return np.sqrt(phi2_corr / max(min(c_corr - 1, r_corr - 1), 1e-12))

tabla = pd.crosstab(analisis["cluster_propiedad"], analisis["segmento_operacion"]).reindex(columns=segmentos, fill_value=0)
v_observada = cramers_v(tabla)
ari_observado = adjusted_rand_score(y, cluster)
ami_observado = adjusted_mutual_info_score(y, cluster)
print("Muestra completa | ARI:", round(ari_observado, 4), "| AMI:", round(ami_observado, 4), "| V de Cramér:", round(v_observada, 4))

# Prueba principal: permutación dentro de fuente × barrio en estratos con varias operaciones.
estrato = (analisis["fuente_norm"].astype(str) + "¦" +
           analisis["barrio_oficial"].fillna("sin_barrio").astype(str)).to_numpy()
grupos = pd.Series(np.arange(len(estrato))).groupby(pd.Series(estrato), sort=False).indices.values()
estratos = [ix for ix in grupos if np.unique(y[ix]).size >= 2]
mask_estrato = np.zeros(len(analisis), dtype=bool)
for ix in estratos:
    mask_estrato[ix] = True
print("Estratos fuente × barrio con más de una operación:", len(estratos))
print("Casos en la prueba principal:", int(mask_estrato.sum()), "de", len(analisis))
if mask_estrato.sum() == 0:
    print("No hay solapamiento suficiente para una prueba condicionada por fuente y barrio.")
    ami_estrato, p_estrato, ami_null_estrato = np.nan, np.nan, np.array([])
else:
    y_estrato, cl_estrato = y[mask_estrato], cluster[mask_estrato]
    ami_estrato = adjusted_mutual_info_score(y_estrato, cl_estrato)
    estrato_sub = estrato[mask_estrato]
    ix_estratos_sub = pd.Series(np.arange(len(estrato_sub))).groupby(pd.Series(estrato_sub), sort=False).indices.values()
    rng_perm = np.random.default_rng(917)
    n_perm = 499
    ami_null_estrato = np.empty(n_perm)
    for b in range(n_perm):
        y_perm = y_estrato.copy()
        for ix in ix_estratos_sub:
            y_perm[ix] = rng_perm.permutation(y_perm[ix])
        ami_null_estrato[b] = adjusted_mutual_info_score(y_perm, cl_estrato)
    p_estrato = (1 + np.sum(ami_null_estrato >= ami_estrato)) / (n_perm + 1)

# Sensibilidad: la permutación solo por fuente conserva menos estructura geográfica.
fuente = analisis["fuente_norm"].astype(str).to_numpy()
ix_fuentes = pd.Series(np.arange(len(fuente))).groupby(pd.Series(fuente), sort=False).indices.values()
fuentes_multi = [ix for ix in ix_fuentes if np.unique(y[ix]).size >= 2]
mask_fuente = np.zeros(len(analisis), dtype=bool)
for ix in fuentes_multi:
    mask_fuente[ix] = True
if mask_fuente.sum():
    y_fuente, cl_fuente = y[mask_fuente], cluster[mask_fuente]
    ami_fuente = adjusted_mutual_info_score(y_fuente, cl_fuente)
    fuente_sub = fuente[mask_fuente]
    ix_fuente_sub = pd.Series(np.arange(len(fuente_sub))).groupby(pd.Series(fuente_sub), sort=False).indices.values
    ami_null_fuente = np.empty(499)
    for b in range(499):
        y_perm = y_fuente.copy()
        for ix in ix_fuente_sub:
            y_perm[ix] = rng_perm.permutation(y_perm[ix])
        ami_null_fuente[b] = adjusted_mutual_info_score(y_perm, cl_fuente)
    p_fuente = (1 + np.sum(ami_null_fuente >= ami_fuente)) / 500
else:
    ami_fuente, p_fuente = np.nan, np.nan

res_contraste = pd.DataFrame([{
    "n_completo": len(analisis), "n_fuente_barrio_comparables": int(mask_estrato.sum()),
    "estratos_fuente_barrio_con_solapamiento": len(estratos),
    "ari_total": ari_observado, "ami_total": ami_observado, "cramers_v_total": v_observada,
    "ami_fuente_barrio": ami_estrato,
    "ami_perm_media_fuente_barrio": float(ami_null_estrato.mean()) if len(ami_null_estrato) else np.nan,
    "ami_perm_p95_fuente_barrio": float(np.quantile(ami_null_estrato, .95)) if len(ami_null_estrato) else np.nan,
    "p_permutacion_fuente_barrio": p_estrato,
    "n_fuentes_comparables": int(mask_fuente.sum()), "ami_fuente": ami_fuente,
    "p_permutacion_fuente": p_fuente, "permutaciones": 499
}])
display(res_contraste.round(4))
res_contraste.to_csv(REPORTS / "pe3_validacion_etiquetas_operacion_v1.csv", index=False)

## 6. Perfil de los grupos y lectura para el caso de negocio

- Se informa la mezcla de operaciones por cluster y la mediana de los atributos que lo definen.
- Precio se incorpora **solo para describir** los grupos ya formados, en tablas separadas por tipo de operación: precio total de venta, precio mensual de alquiler y precio por m² cuando exista. No se interpretan como una escala común.
- Una correspondencia alta indicaría que parte de las diferencias entre mercados publicados está alineada con atributos físicos/urbanos. Una correspondencia baja indicaría que las etiquetas representan modalidades que comparten perfiles o que las variables disponibles no las distinguen.
- La segmentación no demuestra desplazamiento entre alquiler temporal y permanente, demanda insatisfecha, déficit habitacional ni causalidad. Sirve para decidir qué diferencias merecen validación adicional en los indicadores de oferta, accesibilidad y concentración definidos para el Instituto.

### Supuestos y límites de los perfiles

- Las medianas describen los anuncios que componen cada grupo; no representan valores transados ni corrigen la selección de cada portal.
- El precio es suplementario: diferencias observadas por cluster no participaron de su definición y no son efectos atribuibles al grupo.


In [ ]:
perfil_vars = [c for c in features if c in analisis.columns]
perfil = analisis.groupby("cluster_propiedad")[perfil_vars].median().round(3)
perfil.insert(0, "n_publicaciones", analisis.groupby("cluster_propiedad").size())
display(perfil)
perfil.to_csv(REPORTS / "pe3_perfiles_clusters_propiedades_v1.csv")

# Precio como variable suplementaria: resumir por operación sin mezclar unidades.
precios = [c for c in ["precio_venta_usd", "precio_mensual_usd", "precio_m2_usd"] if c in analisis.columns]
if precios:
    filas_precio = []
    for (op_label, cl), grupo in analisis.groupby(["segmento_operacion", "cluster_propiedad"]):
        for variable in precios:
            s = pd.to_numeric(grupo[variable], errors="coerce").dropna()
            if len(s):
                filas_precio.append({"operacion": op_label, "cluster": cl, "variable_precio": variable,
                                     "n": len(s), "mediana": s.median(), "p25": s.quantile(.25), "p75": s.quantile(.75)})
    resumen_precio = pd.DataFrame(filas_precio)
    display(resumen_precio.round(2))
    resumen_precio.to_csv(REPORTS / "pe3_perfil_precios_por_operacion_cluster_v1.csv", index=False)

salida = analisis[["id_publicacion", "fuente_norm", "barrio_oficial", "segmento_operacion", "cluster_propiedad"] + features].copy()
salida.to_csv(config.PROCESSED_DIR / "pe3_clusters_publicaciones_operacion_v1.csv", index=False)


## 7. Criterios para cerrar la interpretación

- Reportar la composición de cada cluster con denominadores y tamaños; revisar clusters dominados por una fuente o barrio.
- Considerar la estabilidad por submuestras y la diferencia entre AMI total y AMI condicionado por fuente.
- Si la permutación condicionada no respalda asociación, no sostener que los segmentos se recuperan naturalmente.
- Si la asociación aparece solo en el contraste total pero desaparece al condicionar por fuente, atribuirla como posible efecto de cobertura/composición del portal.
- Aunque la asociación sea robusta, describirla como correspondencia de perfiles observados y no como explicación causal de la elección de modalidad.
- Antes de la entrega final, ejecutar este notebook después del 12, revisar las variables seleccionadas/cobertura y conservar outputs visibles.

### Supuestos y límites para la decisión

- Alineación entre etiqueta y cluster no significa que el tipo de operación sea explicado por una necesidad habitacional ni que una modalidad desplace a otra.
- La evidencia informa si los anuncios capturados se ordenan de forma parecida en este espacio de atributos; debe leerse junto con KPIs territoriales, censales y de oferta permanente.
- Si falla la cobertura o el soporte común por fuente × barrio, se reporta la limitación y no se reemplaza por una conclusión basada solo en significancia global.


## Panel visual de síntesis: ¿cuánto coinciden perfiles y etiquetas?
Las tarjetas combinan magnitud de asociación y prueba condicionada. El gráfico de composición muestra qué proporción de publicaciones de cada cluster pertenece a cada operación, con el tamaño de grupo. Si la asociación total cae al condicionar por fuente y barrio, parte de la coincidencia puede venir de cómo publican/capturan los portales en cada zona.

In [ ]:
from src.pe3_visuals import kpi_cards
p_txt = f"{p_estrato:.3f}" if pd.notna(p_estrato) else "no estimable"
ami_txt = f"{ami_estrato:.3f}" if pd.notna(ami_estrato) else "no estimable"
kpi_cards([
 ("Publicaciones completas", f"{len(analisis):,}", "tras cobertura y filtros"),
 ("V de Cramér", f"{v_observada:.3f}", "asociación global"),
 ("AMI condicionada", ami_txt, "dentro de fuente × barrio"),
 ("p permutación", p_txt, "nulo condicionado; 499 permutaciones"),
], title="Correspondencia · perfiles y operación",
note="La prueba solo cubre estratos con más de una operación observada.")
prop = tabla.div(tabla.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
fig, ax = plt.subplots(figsize=(8.5,4.5))
colors=["#4b78a8","#6b8f71","#c45a46"]
bottom=np.zeros(len(prop))
for i,col in enumerate(prop.columns):
    vals=prop[col].to_numpy()
    ax.bar(prop.index.astype(str),vals,bottom=bottom,label=str(col),color=colors[i%len(colors)])
    bottom += vals
for i,n_i in enumerate(tabla.sum(axis=1)):
    ax.text(i,102,f"n={int(n_i):,}",ha="center",va="bottom",fontsize=8)
ax.set_ylim(0,115); ax.set_ylabel("Composición de operación (%)")
ax.set_xlabel("Cluster construido sin etiqueta de operación")
ax.set_title("Composición de cada perfil no supervisado")
ax.legend(title="Operación",frameon=False,bbox_to_anchor=(1.02,1),loc="upper left")
for spine in ("top","right"): ax.spines[spine].set_visible(False)
plt.tight_layout(); plt.show()

**Lectura.** La composición describe correspondencia, no “tipos naturales” de vivienda. La evidencia más exigente es la permutación dentro de fuente × barrio; también revisar el tamaño de los estratos incluidos y la sensibilidad por fuente.